In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Random Forests

**Random Forest** is an **ensemble** of Decision Trees, trained using a method called **bagging**.
An **ensemble** is just a group of models that work together to make better predictions.
**Bagging** means that each tree is trained on a random part of the data, which helps reduce overfitting and improve the model’s performance.

Ensemble methods aim to combine the predictions of multiple models to get a more accurate result.

<img src="ensemble.png" width=500 height=200 />

After training all the classifiers, the ensemble makes a prediction for a new instance by **aggregating** their votes. It predicts the class with the most votes.

<img src="aggregation.png" width=500 height=200 />

The decision trees are each trained on different random parts of the training data. When these random samples are taken **with replacement**, it’s called **bagging** (short for bootstrap aggregating).

<img src="bagging.png" width=500 height=200 />

Getting a little bit technical...

Sampling with replacement, or bootstrap sampling, is important because it adds variety to the trees in the Random Forest. Even though all the trees are trained on the same dataset, each one gets a slightly different random sample of the data. This small difference helps to 'decorrelate' the trees, which means they won’t all make the same mistakes or learn the same biases.

When we combine these slightly different trees, such as by letting them vote on a final answer, their errors tend to balance out. This effect—where the differences between the trees make the overall prediction stronger—is why Random Forests are more accurate and reliable than using a single Decision Tree."

In [ ]:
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.tree import plot_tree

from sklearn.inspection import DecisionBoundaryDisplay

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

## Toy 2-dimensional examples

Let’s look at how Decision Trees and Random Forest differ on these 2-dimensional toy datasets. For each model, we’ll plot the classification regions and boundaries.

### Example 1: two-circles dataset

In [ ]:
from sklearn.datasets import make_circles
X, y = make_circles(n_samples=500, factor=0.1, noise=0.4)

In [ ]:
# plot the dataset: inner circle (class 1) + outer circle (class 0)
plt.scatter(X[y==0,0],X[y==0,1], label='class 0')
plt.scatter(X[y==1,0],X[y==1,1], label='class 1')
plt.legend()

In [ ]:
# train/test split
X_train,X_test,y_train,y_test = train_test_split(X,y)

In [ ]:
# Train a Decision Tree with unlimited depth
tree_clf = DecisionTreeClassifier()
tree_clf.fit(X_train, y_train)

In [ ]:
# Plot the decision boundary
DecisionBoundaryDisplay.from_estimator(tree_clf, X,response_method='predict')

# Plot the training points
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor='k', s=20)

This looks like the classification boundaries of an overfitted model.
The boundaries look too complex, trying to capture every small variation.

We can further confirm overfitting by checking the gap between the training and testing accuracy (a big gap usually means the model is performing well on the training data but struggling with new, unseen data).

In [ ]:
# Predictions on training and testing sets
y_train_pred = tree_clf.predict(X_train)
y_test_pred = tree_clf.predict(X_test)

# Calculate and print accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.2f}")
print(f"Testing Accuracy: {test_accuracy:.2f}")

We can also plot the decision tree to see just how complex it is. This is definitely an overly complicated model for such simple data.

In [ ]:
# visualize tree
fig = plt.figure(figsize=(20,15))

plot_tree(tree_clf, class_names =['0','1'], filled=True, impurity=False, fontsize=5)
plt.show()

Let's try using a simpler tree.

In [ ]:
tree_clf = DecisionTreeClassifier(max_depth=1)
tree_clf.fit(X_train,y_train)

In [ ]:
# Predictions on training and testing sets
y_train_pred = tree_clf.predict(X_train)
y_test_pred = tree_clf.predict(X_test)

# Calculate and print accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.2f}")
print(f"Testing Accuracy: {test_accuracy:.2f}")

Now our tree is too simple. There's no gap between the training and testing accuracies, but both are quite low. The model isn’t making great predictions, which we can also see by looking at the classification regions.

In [ ]:
# Plot the decision boundary
DecisionBoundaryDisplay.from_estimator(tree_clf, X,response_method='predict')

# Plot the training points
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor='k', s=20)

In [ ]:
fig = plt.figure(figsize=(10,5))

plot_tree(tree_clf, class_names =['0','1'], filled=True, impurity=False, fontsize=15)

Here’s the magic of ensemble methods! By combining the predictions of hundreds of simple, not-so-accurate models, we can get very accurate results.

In [ ]:
# ensembple of underfitted trees
rf_clf = RandomForestClassifier(n_estimators=100, # number of decision trees
                                    max_depth = 1,  # max depth of the trees
                                    n_jobs = -1)  # use all your CPUs to train the trees

In [ ]:
rf_clf.fit(X_train,y_train)

In [ ]:
# Predictions on training and testing sets
y_train_pred = rf_clf.predict(X_train)
y_test_pred = rf_clf.predict(X_test)

# Calculate and print accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.2f}")
print(f"Testing Accuracy: {test_accuracy:.2f}")

Now, the training and testing accuracies are both solid and comparable, with about 80% of predictions correct. The model is working great!

We can confirm this by checking the classification boundaries. Notice how they now capture the shape of the data, outlining both the inner and outer circles accurately.

In [ ]:
# Plot the decision boundary showing only predictions (class labels)
DecisionBoundaryDisplay.from_estimator(rf_clf, X,response_method='predict')

# Plot the training points
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor='k', s=20)

In [ ]:
# ensembple of overfitted trees
rf_clf = RandomForestClassifier(n_estimators=100, # number of decision trees
                                    max_depth = None,  # max depth of the trees
                                    n_jobs = -1)  # use all your CPUs to train the trees

In [ ]:
rf_clf.fit(X_train,y_train)

In [ ]:
# Predictions on training and testing sets
y_train_pred = rf_clf.predict(X_train)
y_test_pred = rf_clf.predict(X_test)

# Calculate and print accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.2f}")
print(f"Testing Accuracy: {test_accuracy:.2f}")

In [ ]:
# Plot the decision boundary showing only predictions (class labels)
DecisionBoundaryDisplay.from_estimator(rf_clf, X,response_method='predict')

# Plot the training points
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor='k', s=20)

### Example 2: two-moons dataset

Let's try using a Random Forest on a different dataset.

In [ ]:
from sklearn.datasets import make_moons
X,y = make_moons(n_samples=1000, noise=0.2)

In [ ]:
plt.scatter(X[y==0,0],X[y==0,1], label='class 0')
plt.scatter(X[y==1,0],X[y==1,1], label='class 1')
plt.legend()

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y)

In [ ]:
# Hyperparameters chosen through a bit of trial and error
rf_clf = RandomForestClassifier(n_estimators=200,  # number of trees
                                max_depth=5,       # maximum depth of each tree
                                n_jobs=-1)         # use all CPUs to train the trees
rf_clf.fit(X_train, y_train)

In [ ]:
# Predictions on training and testing sets
y_train_pred = rf_clf.predict(X_train)
y_test_pred = rf_clf.predict(X_test)

# Calculate and print accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.2f}")
print(f"Testing Accuracy: {test_accuracy:.2f}")

In [ ]:
# Plot the decision boundary showing only predictions (class labels)
DecisionBoundaryDisplay.from_estimator(rf_clf, X,response_method='predict')

# Plot the training points
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor='k', s=20)

## Example 3: regression

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:

# 1D input so we can plot y(x)
n = 150
X = np.linspace(-3.5, 3.5, n).reshape(-1, 1)
y = np.sin(1.5 * X).flatten()
noise = 0.25*np.random.randn(n)
y = y + noise

# Add a few outliers
idx_out = np.random.choice(n, size=15, replace=False)
y[idx_out] += 3*np.random.rand(len(idx_out))
plt.scatter(X,y)

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y)

In [ ]:
tree1 = DecisionTreeRegressor(max_depth=3)
tree2  = DecisionTreeRegressor(max_depth=None)
rf1  = RandomForestRegressor(
    n_estimators=200,
    max_depth=3,
)

In [ ]:
tree1.fit(X_train,y_train)
tree2.fit(X_train,y_train)
rf1.fit(X_train,y_train)
rf2.fit(X_train,y_train)

In [ ]:
# Grid for smooth curves
X_plot = np.linspace(X.min(), X.max(), 800).reshape(-1, 1)

In [ ]:
# tree 1 plot
y_plot = tree1.predict(X_plot)
plt.scatter(X,y)
plt.plot(X_plot,y_plot,color='red')

In [ ]:
y_test_pred = tree1.predict(X_test)
mean_squared_error(y_test,y_test_pred)

In [ ]:
# tree 2 plot
y_plot = tree2.predict(X_plot)
plt.scatter(X,y)
plt.plot(X_plot,y_plot,color='red')

In [ ]:
y_test_pred = tree2.predict(X_test)
mean_squared_error(y_test,y_test_pred)

In [ ]:
# random forest plot
y_plot = rf1.predict(X_plot)
plt.scatter(X,y)
plt.plot(X_plot,y_plot,color='red')

In [ ]:
# random forest plot
y_plot = rf2.predict(X_plot)
plt.scatter(X,y)
plt.plot(X_plot,y_plot,color='red')

## Random Forest Grid Search

Random Forests have several important hyperparameters that control the complexity of their trees and others that affect the overall ensemble. Here are some key ones:

- `min_samples_split`: The minimum number of samples needed to split a node. Lower values create deeper, more complex trees.
- `min_samples_leaf`: The minimum number of samples required to be at a leaf node. Higher values help reduce overfitting by smoothing the model.
- `max_leaf_nodes`: Limits the maximum number of leaf nodes in each tree, controlling the tree size.
- `max_features`: The maximum number of features considered when looking for the best split at each node. Setting this lower than the total feature count encourages diversity among the trees, which improves overall performance.

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
# load iris data
path = 'https://raw.githubusercontent.com/um-perez-alvaro/Data-Science-Practice/master/Data/iris.csv'
df = pd.read_csv(path)
df.head()

In [ ]:
X = df[['sepal_length','sepal_width','petal_length','petal_width']]
y = df.species

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y)

In [ ]:
rf_clf = RandomForestClassifier(n_jobs=-1)

In [ ]:
# Some parameters to try for tuning:
parameters = {
    "min_samples_leaf": [1, 4, 8, 16],       # default is 1
    "min_samples_split": [2, 4, 8, 16],      # default is 2
    "n_estimators": [25, 50, 100, 150, 200], # default is 100
    "max_depth": [2, 5, 10]                  # default is None (unlimited)
}

grid = GridSearchCV(rf_clf, parameters, cv=5, scoring='accuracy', n_jobs=-1, verbose=True)
grid.fit(X_train, y_train)

In [ ]:
grid.best_score_

In [ ]:
grid.best_params_

In [ ]:
best_rf_clf = grid.best_estimator_

In [ ]:
# Predictions on training and testing sets
y_train_pred = best_rf_clf.predict(X_train)
y_test_pred = best_rf_clf.predict(X_test)

# Calculate and print accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.2f}")
print(f"Testing Accuracy: {test_accuracy:.2f}")

Not too bad!

## Feature Importance in Random Forests

One big advantage of Random Forests is their ability to estimate the importance of each feature for making predictions. This gives us insight into which features are most influential in predicting the target variable.

The idea behind feature importance is to look at how often and how early a feature is used to split data in the individual trees. Features that appear closer to the top of the tree affect more samples because they are considered first, impacting a larger portion of the data. As a result, these features contribute more to the model’s decisions and are seen as more 'important' for making accurate predictions.

The `.feature_importances_` method returns the relative importance of each feature.

In [ ]:
# Create a DataFrame for feature importances and plot them
feature_importances = pd.DataFrame(data=best_clf.feature_importances_, index=X.columns, columns=['Feature Importance'])
feature_importances.plot.barh()

Based on these feature importances, sepal length and sepal width appear to be the most important features for making predictions. In contrast, sepal width seems to play only a small role in the model’s predictions.

Feature importances in Random Forests can be a bit controversial.
The `.feature_importances_` method measures importance based on the position of features in the trees, which can sometimes be biased, especially if there are highly correlated features.

An alternative is **permutation feature importance**, which provides a more reliable measure by randomly shuffling each feature and seeing how it impacts the model's accuracy. 
This method directly evaluates how much each feature contributes to predictions by observing the drop in accuracy when the feature’s values are randomized.

Here’s how to calculate permutation feature importances using `sklearn`:

In [ ]:
from sklearn.inspection import permutation_importance

# Compute permutation importances
perm_importance = permutation_importance(best_rf_clf, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

# Create a DataFrame for permutation importances and plot them
perm_importances_df = pd.DataFrame(data=perm_importance.importances_mean, index=X.columns, columns=['Permutation Importance'])
perm_importances_df.plot.barh()

You can use permutation importances with any model, not just Random Forests, and they work for both classification and regression problems.